# The FT-Transformer teammate's 12 runs, in Colab

Runs `runs.py` from this Google Drive folder on a Colab GPU. Every result is saved back to Drive.

1. **Runtime > Change runtime type > A100 GPU** (or L4). Use the same GPU type for all 3 seeds of a model.
2. **Runtime > Run all.** This runs the checks and the timing test only; nothing is trained.
3. Then set `TRAIN = True` in section 3 and run that cell. Keep this tab open.

If Colab disconnects, run the cells again. Finished runs are skipped, and the interrupted run continues from its last saved epoch, if the GPU type is the same.

## Setup

Connects Google Drive and moves into the project folder.

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/IEEE Fraud Detection"   # the folder that holds runs.py
os.chdir(PROJECT)
print(sorted(os.listdir()))

Mounted at /content/drive
['README.md', 'colab.ipynb', 'common.py', 'data', 'models.py', 'references', 'reports', 'runs.py']


In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import platform, torch, pandas, pyarrow, sklearn
print("python", platform.python_version(), "| torch", torch.__version__, "| pandas", pandas.__version__,
      "| pyarrow", pyarrow.__version__, "| scikit-learn", sklearn.__version__)

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
python 3.13.16 | torch 2.14.1+cu130 | pandas 2.2.3 | pyarrow 23.0.1 | scikit-learn 1.6.1


## 1. Checks (trains nothing)

The split file and the graph files must be the team's files (same SHA-256), the column sets must have 390 and 51 columns, and the graph must line up with the split. Writes `reports/checks.md`.

In [3]:
!python runs.py checks

# Data and graph checks

Written by `python runs.py checks`. Every line is an `assert` in common.py that passed.

## Split file and column sets

- Split file `ieee_sorted_split.parquet`: SHA-256 matches the team's file; 590,540 payments sorted by TransactionDT.

| split | payments | frauds | fraud rate | days |
|---|---|---|---|---|
| train | 413,378 | 14,538 | 3.52% | 1.0-120.8 |
| val | 88,581 | 3,042 | 3.43% | 120.8-152.2 |
| test | 88,581 | 3,083 | 3.48% | 152.2-183.0 |

- Column set A: 390 columns (369 numeric, 21 categorical).
- Column set B: 51 columns (30 numeric, 21 categorical).
- Numeric columns with one value in training (divided by 1, not 0): V107.

## Graph files

- Graph file `graph_files.zip`: SHA-256 matches the team's file; 14,239 nodes, as graph_info.json says.
- Real links: 336,236 (each link once per direction, no self-links, never two entities of the same type); 12,401,340 before repeats removed = 413,378 x 30.
- Payment node files line up with the split file row 

## 2. Timing on this GPU (saves nothing)

Trains 10 batches of each model twice. It prints the time per epoch, the peak GPU memory, and whether two identical runs give identical scores.

In [4]:
!python runs.py smoke

{'gpu': 'NVIDIA A100-SXM4-80GB', 'torch': '2.14.1+cu130', 'cuda': '13.0', 'python': '3.13.16'}
GraphSAGE: 0.04 s per batch -> about 0.1 min per epoch | peak GPU 0.4 GB | two identical runs: identical
FT-Transformer B (1 pieces per batch): 0.08 s per batch -> about 0.3 min per epoch | peak GPU 3.5 GB | two identical runs: identical
FT-Transformer A (2 pieces per batch): 0.88 s per batch -> about 3.1 min per epoch | peak GPU 12.7 GB | two identical runs: identical


## 3. Training: the 12 runs

The order is GraphSAGE, then shuffled GraphSAGE, then FT-Transformer on set B, then on set A. Each run's output is also written to `results/graph_vs_V/logs/`.

In [7]:
TRAIN = True   # set to True after the checks and the timing test above look right
if TRAIN:
    !python runs.py all

[sage_seed42] GraphSAGE on NVIDIA A100-SXM4-80GB
  epoch  1 | loss 0.4311 | val ROC-AUC 0.7885 | val PR-AUC 0.1819 | 9 s
  epoch  2 | loss 0.3657 | val ROC-AUC 0.7890 | val PR-AUC 0.1903 | 8 s
  epoch  3 | loss 0.3493 | val ROC-AUC 0.7803 | val PR-AUC 0.1899 | 8 s
  epoch  4 | loss 0.3404 | val ROC-AUC 0.7760 | val PR-AUC 0.1945 | 8 s
  epoch  5 | loss 0.3342 | val ROC-AUC 0.7803 | val PR-AUC 0.2038 | 8 s
  epoch  6 | loss 0.3323 | val ROC-AUC 0.7768 | val PR-AUC 0.2080 | 8 s
  epoch  7 | loss 0.3274 | val ROC-AUC 0.7740 | val PR-AUC 0.2088 | 8 s
  epoch  8 | loss 0.3240 | val ROC-AUC 0.7754 | val PR-AUC 0.2036 | 8 s
  epoch  9 | loss 0.3237 | val ROC-AUC 0.7773 | val PR-AUC 0.2169 | 8 s
  epoch 10 | loss 0.3190 | val ROC-AUC 0.7736 | val PR-AUC 0.2175 | 8 s
  epoch 11 | loss 0.3164 | val ROC-AUC 0.7760 | val PR-AUC 0.2135 | 8 s
  epoch 12 | loss 0.3131 | val ROC-AUC 0.7778 | val PR-AUC 0.2273 | 8 s
[sage_seed42] test ROC-AUC 0.7538 | test PR-AUC 0.1406  -> results/graph_vs_V/runs.csv


## 4. Finished runs

One row per finished run, from `results/graph_vs_V/runs.csv`.

In [6]:
import pandas as pd

runs_csv = "results/graph_vs_V/runs.csv"
pd.read_csv(runs_csv) if os.path.exists(runs_csv) else "no finished runs yet"

'no finished runs yet'